In [ ]:
#!pip install imitation

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from torch.utils.data import Dataset

In [3]:
class ExpertDataset(Dataset):
    def __init__(self, observations, actions):
        self.observations = observations
        self.actions = actions

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        return self.observations[idx], self.actions[idx]

In [34]:
from imitation.data.types import Transitions
from imitation.algorithms.bc import BC
from stable_baselines3.common.env_util import make_vec_env

In [8]:
import numpy as np
from apad_puzzle_rl.envs.apad_env import APADEnv
env = APADEnv()

In [10]:
data = np.load("demo.npz")
observations = data["observations"]
actions = data["actions"]
episode_starts = data["episode_starts"]
dones = np.zeros(len(observations), dtype=bool)
dones[7] = True
dones[15] = True

In [23]:
import imitation
print(imitation.__version__)

1.0.1


In [26]:
next_obs = np.vstack([observations[1:], observations[-1:]])  # repeat last obs as dummy

# Define transitions
transitions = Transitions(
    obs=observations,
    acts=actions,
    dones=dones,
    next_obs=next_obs,
    infos=[{} for _ in range(len(actions))],  # dummy
)

# Create a BC model
bc_trainer = BC(
    observation_space=env.observation_space,
    action_space=env.action_space,
    demonstrations=transitions,
    rng=np.random.default_rng(0),
    batch_size=16,
    l2_weight=0.0,
)

# Train
bc_trainer.train(n_epochs=10)

0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 16       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00792 |
|    entropy        | 7.92     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 129      |
|    loss           | 7.91     |
|    neglogp        | 7.92     |
|    prob_true_act  | 0.000363 |
|    samples_so_far | 16       |
--------------------------------



Epoch 0 of 10           
Epoch 1 of 10
Epoch 2 of 10
Epoch 3 of 10
Epoch 4 of 10
Epoch 5 of 10
Epoch 6 of 10
Epoch 7 of 10
Epoch 8 of 10
10batch [00:00, 168.46batch/s]


In [27]:
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback

In [28]:
class TimerCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.start_time = time.time()
    
    def _on_step(self):
        if self.num_timesteps % 1000 == 0:
            elapsed = time.time() - self.start_time
            rate = self.num_timesteps / elapsed
            remaining = (self.locals['total_timesteps'] - self.num_timesteps) / rate
            print(f"Step {self.num_timesteps}, {elapsed:.0f}s elapsed, {remaining:.0f}s remaining")
        return True

class GradNormCallback(BaseCallback):
    def _on_step(self):
        if hasattr(self.model.policy, 'parameters'):
            total_norm = 0
            for p in self.model.policy.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** (1. / 2)
            
            self.logger.record("train/grad_norm", total_norm)
        return True

In [38]:
import time
from stable_baselines3.common.env_util import make_vec_env

In [39]:
env = make_vec_env(lambda: APADEnv(), n_envs=1)
env.reset()
model = None
model = MaskablePPO(
    "MlpPolicy",
    env,
    n_steps = 512,
    tensorboard_log="./maskable_ppo_logs_6/",
    verbose=1,
    policy_kwargs = dict(net_arch=[32, 32])
)
model.policy.load_state_dict(bc_trainer.policy.state_dict())

Using cpu device


<All keys matched successfully>

In [40]:
model.learn(total_timesteps=25000, reset_num_timesteps=True, callback=[TimerCallback(), GradNormCallback()])

Logging to ./maskable_ppo_logs_6/PPO_4


AttributeError: 'DummyVecEnv' object has no attribute 'has_attr'

In [ ]:
model.save(f"mppo_model_5")